# Géocodage SITG-labs

## Vérifier json API

In [1]:
from sitg_geocode import inspect_sitg_response

await inspect_sitg_response("Av. J-D Maillard, 7, Meyrin")

{
  "hits": [
    {
      "IDPADR": "930225017479",
      "ADRESSE": "Avenue Jacob-Daniel-MAILLARD 7",
      "TYPE": "Existante",
      "TYVOIE": "Avenue",
      "TYPABR": "av.",
      "LIANT": "Jacob-Daniel-",
      "NOMVOI": "MAILLARD",
      "NO_ADRESSE": "7",
      "COMMUNE": "Meyrin",
      "NO_POSTAL": "1217",
      "NOM_NPA": "Meyrin",
      "EGID": "1021280",
      "EGRID": "CH276384676537",
      "longitude": 6.068229,
      "latitude": 46.22923,
      "easting": 2494278.54,
      "northing": 1120679.36,
      "indexed_at": "2026-05-08T06:02:03.508686+00:00",
      "analyzed_NOM_NPA": "meyrin",
      "analyzed_COMMUNE": "meyrin",
      "analyzed_NOMVOI": "maillard",
      "analyzed_TYVOIE": "avenue",
      "phonetized_analyzed_NOMVOI": "MAIAR",
      "score": 78.43
    }
  ],
  "offset": 0,
  "limit": 1,
  "nbHits": 1,
  "exhaustiveNbHits": false,
  "processingTimeMs": 1,
  "query": "maillard"
}


## Corriger liste d'adresses

In [4]:
import polars as pl

from sitg_geocode import inspect_sitg_response, sitg_geocode_async

# lire le CSV de test
df = pl.read_csv("test_adresses.csv")

# Geocodage
result_geocode = await sitg_geocode_async(
    df,
    col_adresse="Rue et N°",
    output_format="polars",
    max_concurrent = 10,
    min_score_threshold=95)

# Joindre et exporter
df_resultat = df.join(result_geocode, on="Rue et N°", how="left")
df_resultat.write_csv("resultat.csv")

print(df_resultat.head(5))

Geocoding SITG:   0%|          | 0/107 [00:00<?, ?it/s]

shape: (5, 12)
┌────────────┬────────────┬──────────┬───────────┬───┬──────────┬──────────┬───────────┬───────────┐
│ Rue et N°  ┆ SITG_ADRES ┆ SITG_NPA ┆ SITG_NOM_ ┆ … ┆ SITG_LON ┆ SITG_LAT ┆ SITG_EST_ ┆ SITG_NORD │
│ ---        ┆ SE         ┆ ---      ┆ NPA       ┆   ┆ ---      ┆ ---      ┆ EPSG_2056 ┆ _EPSG_205 │
│ str        ┆ ---        ┆ i64      ┆ ---       ┆   ┆ f64      ┆ f64      ┆ ---       ┆ 6         │
│            ┆ str        ┆          ┆ str       ┆   ┆          ┆          ┆ f64       ┆ ---       │
│            ┆            ┆          ┆           ┆   ┆          ┆          ┆           ┆ f64       │
╞════════════╪════════════╪══════════╪═══════════╪═══╪══════════╪══════════╪═══════════╪═══════════╡
│ 12 rue Jea ┆ Rue Jean-C ┆ 1202     ┆ Genève    ┆ … ┆ 6.148414 ┆ 46.21442 ┆ 2.5004e6  ┆ 1.1189e6  │
│ n-Charles  ┆ harles-AMA ┆          ┆           ┆   ┆          ┆          ┆           ┆           │
│ Amat       ┆ T 12       ┆          ┆           ┆   ┆          ┆          ┆

In [15]:
with pl.Config() as cfg:
    cfg.set_tbl_formatting("MARKDOWN"),
    cfg.set_tbl_hide_column_data_types(True),
    cfg.set_tbl_hide_dataframe_shape(True),
    cfg.set_tbl_cols(-1)
    print(df_resultat.head(5))

| Rue et | SITG_A | SITG_N | SITG_ | SITG_ | SITG_ | SITG_ | SITG_ | SITG_ | SITG_ | SITG_ | SITG_ |
| N°     | DRESSE | PA     | NOM_N | COMMU | EGID  | EGRID | SCORE | LON   | LAT   | EST_E | NORD_ |
|        |        |        | PA    | NE    |       |       |       |       |       | PSG_2 | EPSG_ |
|        |        |        |       |       |       |       |       |       |       | 056   | 2056  |
|--------|--------|--------|-------|-------|-------|-------|-------|-------|-------|-------|-------|
| 12 rue | Rue    | 1202   | Genèv | Genèv | 20377 | CH168 | 95.83 | 6.148 | 46.21 | 2.500 | 1.118 |
| Jean-C | Jean-C |        | e     | e-Pet | 21    | 16563 |       | 414   | 442   | 4e6   | 9e6   |
| harles | harles |        |       | it-Sa |       | 8919  |       |       |       |       |       |
| Amat   | -AMAT  |        |       | conne |       |       |       |       |       |       |       |
|        | 12     |        |       | x     |       |       |       |       |       |       